# OncoGO-CRC — Cuaderno reproducible (Google Colab)

**Objetivo de la tesis (resumen):** integrar de forma reproducible proteómica pública de cáncer colorrectal (CRC): **CPTAC/PDC** como fuente primaria, **PRIDE** como validación en tejido, **plasma** como capa clínica, análisis centrado en **Gene Ontology**, redes, **machine learning explicable** y clasificación por evidencia.

Este cuaderno te permite **clonar el repositorio**, **instalar el paquete**, **descargar datos reales** (PDC/PRIDE), **harmonizar metadata**, **validar**, **versionar** y **ejecutar el pipeline** paso a paso, con explicaciones en cada bloque.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JancoDeveloper/OncoGO-CRC/blob/main/notebooks/oncogo_crc_colab_tesis.ipynb)

> **Compartir:** Archivo → *Compartir* (Colab) o abre el enlace de arriba cuando el notebook esté en GitHub (`main`).

> **Requisitos:** tiempo de ejecución con **internet**; opcionalmente un token **PDC** (`PDC_TOKEN`) si el portal lo exige para GraphQL; RAM suficiente (datasets grandes).


## 0. Entorno y rutas en Colab

En Colab el directorio de trabajo suele ser `/content`. La variable `USE_GOOGLE_DRIVE` (celda siguiente) permite **montar Drive** y clonar el repo en `MyDrive/OncoGO-CRC-colab/OncoGO-CRC` para **no perder** descargas y resultados al cerrar la sesión. El resto de celdas usa la variable de entorno `ONCOGO_ROOT`.


In [ ]:
import sys
import os
from pathlib import Path

# True = montar Google Drive y usar MyDrive/OncoGO-CRC-colab/ (persiste entre sesiones de Colab)
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    work = Path("/content/drive/MyDrive/OncoGO-CRC-colab")
    work.mkdir(parents=True, exist_ok=True)
    ROOT = work / "OncoGO-CRC"
else:
    ROOT = Path("/content/OncoGO-CRC")

os.environ["ONCOGO_ROOT"] = str(ROOT)
print("Python:", sys.version)
print("ROOT:", ROOT)
print("ONCOGO_ROOT:", os.environ["ONCOGO_ROOT"])


## 1. Clonar el repositorio

El destino del clone es **`ONCOGO_ROOT`** (celda 0): `/content/OncoGO-CRC` o, si activaste Drive, `MyDrive/OncoGO-CRC-colab/OncoGO-CRC`. Cambia `REPO_URL` si usas otro fork. Si el repo es **privado**, configura un [token de GitHub](https://github.com/settings/tokens) en la URL vía *Colab secrets* (no lo pegues en texto claro en cuadernos públicos).


In [ ]:
# --- CONFIGURA AQUÍ ---
REPO_URL = "https://github.com/JancoDeveloper/OncoGO-CRC.git"  # <-- cambia si usas otro fork
BRANCH = "main"  # o master, o el nombre de tu rama

import os
import subprocess
from pathlib import Path

ROOT = Path(os.environ.get("ONCOGO_ROOT", "/content/OncoGO-CRC"))

if (ROOT / ".git").exists():
    print("Repo ya clonado en:", ROOT)
elif ROOT.exists():
    raise SystemExit(
        f"Existe {ROOT} pero no es un repo git. Borra o renombra la carpeta y vuelve a ejecutar esta celda."
    )
else:
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(ROOT)],
        check=True,
    )
    print("Clone OK en", ROOT)


## 2. Instalar dependencias y el paquete `oncogo-crc`

Instalamos dependencias científicas y el proyecto en modo editable (`-e`) para que el comando `oncogo-crc` esté disponible. Opcionalmente se añaden extras para GO remoto y explicabilidad (`shap`).


In [ ]:
import os
from pathlib import Path
ROOT = Path(os.environ.get("ONCOGO_ROOT", "/content/OncoGO-CRC"))
os.chdir(ROOT)

!pip install -q -U pip setuptools wheel
!pip install -q -e ".[go-remote,explain]"

# Comprobar CLI
!oncogo-crc --help


## 3. (Opcional) Token PDC y otros secretos

El portal **PDC** a veces requiere autenticación para `filesPerStudy`. En Colab: *🔑 Secretos* → añade `PDC_TOKEN` (si aplica). El `config/config.yaml` ya referencia la variable de entorno.

Si no usas secretos, la siguiente celda no hace nada.


In [ ]:
try:
    from google.colab import userdata
    import os
    tok = userdata.get("PDC_TOKEN")
    if tok:
        os.environ["PDC_TOKEN"] = tok
        print("PDC_TOKEN cargado desde Colab secrets.")
    else:
        print("No hay secret PDC_TOKEN (opcional).")
except ImportError:
    print("No estás en Colab o no hay userdata; define manualmente: import os; os.environ['PDC_TOKEN']='...'")


## 4. ¿Qué hace `download_real`?

**Qué se busca:** bajar matrices y metadatos **procesados** (TSV/CSV/xlsx), no RAW de MS, según `config/datasets.yaml`: CPTAC vía **PDC** (estudio `PDC000116`), PRIDE **PXD034575**, plasma **PXD036594**.

**Qué hace:** resuelve URLs (GraphQL PDC, API PRIDE), descarga a `data/raw/...`, calcula SHA-256 y escribe manifiestos en `data/raw/download_real_manifest.tsv` y `data/manifests/`.


In [ ]:
import os
from pathlib import Path
ROOT = Path(os.environ.get("ONCOGO_ROOT", "/content/OncoGO-CRC"))
os.chdir(ROOT)
!oncogo-crc run --stage download_real


## 5. `harmonize_public_metadata` — alinear muestras con la matriz

**Qué se busca:** que cada columna de la matriz CPTAC (TMT) tenga fila de metadata con el mismo `sample_id`; recortar PRIDE a columnas LFQ + `protein_id`; preparar plasma (péptidos + metadata provisional si aplica).

**Qué hace:** descarga mapping/biospecimen/clínica PDC, regenera `metadata/sample_metadata/cptac_metadata.tsv`, etc. (ver código en `src/oncogo_crc/utils/public_metadata_harmonize.py`).


In [ ]:
import os
from pathlib import Path
ROOT = Path(os.environ.get("ONCOGO_ROOT", "/content/OncoGO-CRC"))
os.chdir(ROOT)
!oncogo-crc run --stage harmonize_public_metadata


## 6. `validate_inputs` — comprobar coherencia

**Qué se busca:** matriz con `protein_id` en primera columna, metadata con `sample_id` y `condition`, cruce exacto con columnas, y que las comparaciones de `config/comparisons.yaml` tengan sentido.

**Salida:** `results/validation/input_validation_report.json`.


In [ ]:
import os
from pathlib import Path
ROOT = Path(os.environ.get("ONCOGO_ROOT", "/content/OncoGO-CRC"))
os.chdir(ROOT)
!oncogo-crc run --stage validate_inputs


## 7. `version_inputs` — congelar versión de datos

**Qué se busca:** trazabilidad para la memoria (qué copia de insumos se usó).

**Salida:** carpeta bajo `data/versions/<timestamp>/`.


In [ ]:
import os
from pathlib import Path
ROOT = Path(os.environ.get("ONCOGO_ROOT", "/content/OncoGO-CRC"))
os.chdir(ROOT)
!oncogo-crc run --stage version_inputs


## 8. Pipeline analítico completo (opcional)

**Qué se busca:** desde IDs harmonizados → QC → diferencial → GO → red → ML → informe.

**Nota:** puede tardar y mostrar avisos de SciPy (“catastrophic cancellation”) en proteínas con varianza casi nula; no siempre es error.

`ingest_cptac` **no** entra en `--stage all`; se invoca aparte si quieres la vía `cptac` Python.


In [ ]:
import os
from pathlib import Path
ROOT = Path(os.environ.get("ONCOGO_ROOT", "/content/OncoGO-CRC"))
os.chdir(ROOT)
# Ejecuta todas las etapas del pipeline (excepto ingest_cptac)
!oncogo-crc run --stage all


## 9. Regenerar páginas web resumidas (MkDocs / Markdown)

**Qué se busca:** un sitio legible con tablas resumidas e inventario de `results/` (ver `src/oncogo_crc/utils/web_publish.py`).

Tras esto puedes instalar MkDocs y generar `site/` o leer los `.md` en `docs/web/`.


In [ ]:
import os
from pathlib import Path
ROOT = Path(os.environ.get("ONCOGO_ROOT", "/content/OncoGO-CRC"))
os.chdir(ROOT)
!oncogo-crc run --stage publish_results
!pip install -q mkdocs mkdocs-material
!mkdocs build -q
print("Sitio estático en:", ROOT / "site/index.html")


## 10. (Opcional) Empaquetar `results/` y `reports/` para descarga

Genera zips en `/content/oncogo_crc_results.zip` y `/content/oncogo_crc_reports.zip` (siempre bajo `/content/`) para descargarlos fácil desde el panel de Colab (*icono de carpeta*), aunque el proyecto esté en Drive.


In [ ]:
import shutil
from pathlib import Path

import os
base = Path(os.environ.get("ONCOGO_ROOT", "/content/OncoGO-CRC"))

out_results = Path("/content/oncogo_crc_results.zip")
if (base / "results").exists():
    if out_results.exists():
        out_results.unlink()
    shutil.make_archive(str(out_results.with_suffix("")), "zip", root_dir=base, base_dir="results")
    print("Descarga:", out_results)
else:
    print("No existe aún results/; ejecuta el pipeline antes.")

out_reports = Path("/content/oncogo_crc_reports.zip")
if (base / "reports").exists():
    if out_reports.exists():
        out_reports.unlink()
    shutil.make_archive(str(out_reports.with_suffix("")), "zip", root_dir=base, base_dir="reports")
    print("Descarga:", out_reports)


## 11. Exploración rápida con pandas

Abre un resultado diferencial y muestra las primeras filas ordenadas por `padj`.


In [ ]:
import os
import pandas as pd
from pathlib import Path

ROOT = Path(os.environ.get("ONCOGO_ROOT", "/content/OncoGO-CRC"))
p = ROOT / "results/differential/cptac_tumor_vs_normal_diff.tsv"
if p.exists():
    df = pd.read_csv(p, sep="\t")
    print(df.shape)
    display(df.sort_values("padj").head(15))
else:
    print("Aún no existe el archivo; ejecuta antes el pipeline o la etapa differential.")


## Referencias útiles en el repo

| Ruta | Contenido |
|------|-----------|
| `DOCUMENTACION_TESIS.md` | Visión general del pipeline |
| `config/datasets.yaml` | Datasets y descargas |
| `config/comparisons.yaml` | Comparaciones tumor/normal, plasma, PRIDE |
| `README.md` | CLI, MkDocs, publicación |

---

**Fin del cuaderno.** Si compartes el Colab, recuerda no incluir tokens en texto plano; usa *Secretos* de Colab o variables de entorno privadas.
